In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# Phase 9 — Cluster Interpretation, Business Implications & Limitations

**Project:** Wholesale Customer Segmentation

Phase 9 - Interpretation, Implications & Limitations
Wholesale Customers Clustering Analysis

Steps covered (per implementation plan):
 24. Interpret each cluster
 25. Explain business implications
 26. Evaluate limitations
 27. Write conclusion

Depends on: phase6_train_validate.py output (labeled_customers.csv),
            findings established in Phase 7 (profiles, ANOVA ranking,
            Channel cross-tab) and Phase 8 (PCA variance/loadings)
Outputs: section9_interpretation.md (report-ready text), printed derivation

Design note: cluster names and business claims below are DERIVED from the
actual labeled data in this script (re-computed, not hard-coded from memory
of earlier phases' printed output), so the interpretation is verifiably
grounded in real cluster output rather than pre-assumed — per the plan's
explicit instruction.

### How to use this notebook
Run cells from top to bottom. Keep the project files in the same folder as this notebook. Phases 1–4 create the data preparation artifacts used by later phases; Phases 5–9 read those artifacts; Phase 10 assembles the final report.

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import pandas as pd



# Recreate constants locally. Phase 6 supplies labeled_customers.csv.
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

PRIMARY_CLUSTER_COL = "cluster_k2"
CHANNEL_MAP = {1: "Horeca (1)", 2: "Retail (2)"}

## SECTION 1: Load Labeled Data & Recompute Grounding Statistics

In [ ]:
# SECTION 1: Load Labeled Data & Recompute Grounding Statistics
# ===========================================================================
def load_and_recompute_stats() -> dict:
    """Load labeled data and recompute the exact statistics needed to name
    and interpret clusters, so interpretation is traceable to real numbers
    rather than restated from memory of earlier phases."""
    print("=" * 70)
    print("SECTION 1: LOAD LABELED DATA & RECOMPUTE GROUNDING STATISTICS")
    print("=" * 70)

    df = pd.read_csv("labeled_customers.csv")
    mean_profile = df.groupby(PRIMARY_CLUSTER_COL)[SPEND_COLS].mean()
    sizes = df[PRIMARY_CLUSTER_COL].value_counts().sort_index()
    pct = (sizes / len(df) * 100).round(1)

    channel_ct_pct = pd.crosstab(
        df[PRIMARY_CLUSTER_COL], df["Channel"].map(CHANNEL_MAP), normalize="index"
    ) * 100

    total_spend = mean_profile.sum(axis=1)

    print(f"Loaded {len(df)} labeled customers.")
    print(f"\nCluster sizes: {sizes.to_dict()} (% of total: {pct.to_dict()})")
    print(f"\nMean spend per category per cluster:\n{mean_profile.round(1).to_string()}")
    print(f"\nMean total spend per cluster:\n{total_spend.round(1).to_string()}")
    print(f"\nChannel composition (row %):\n{channel_ct_pct.round(1).to_string()}")

    return {
        "df": df,
        "mean_profile": mean_profile,
        "sizes": sizes,
        "pct": pct,
        "channel_ct_pct": channel_ct_pct,
        "total_spend": total_spend,
    }

## SECTION 2: Name & Interpret Each Cluster (Step 24)

In [ ]:
# SECTION 2: Name & Interpret Each Cluster (Step 24)
# ===========================================================================
def name_and_interpret_clusters(stats: dict) -> dict:
    """Derive a plain-business-language name for each cluster from its
    actual dominant spend categories and channel composition — not chosen
    in advance. Naming rule (applied mechanically, not by eyeballing):
    for each cluster, identify its top-2 spend categories by mean value,
    and its majority channel, and compose a name from those facts."""
    print("\n" + "=" * 70)
    print("SECTION 2: NAME & INTERPRET EACH CLUSTER")
    print("=" * 70)

    mean_profile = stats["mean_profile"]
    channel_ct_pct = stats["channel_ct_pct"]
    sizes = stats["sizes"]
    pct = stats["pct"]
    total_spend = stats["total_spend"]

    cluster_names = {}
    interpretations = {}

    for cluster_id in mean_profile.index:
        top2 = mean_profile.loc[cluster_id].sort_values(ascending=False).index[:2].tolist()
        majority_channel = channel_ct_pct.loc[cluster_id].idxmax()
        majority_channel_pct = channel_ct_pct.loc[cluster_id].max()
        spend_level = "higher-total-spend" if total_spend[cluster_id] == total_spend.max() else "lower-total-spend"

        # Names are based only on clustering variables; Channel is reported separately.
        name = f"{spend_level.title()} {top2[0]}/{top2[1]}-Oriented Segment"
        cluster_names[cluster_id] = name

        interpretation = (
            f"Cluster {cluster_id} — \"{name}\"\n"
            f"  Size: {sizes[cluster_id]} customers ({pct[cluster_id]}% of the dataset).\n"
            f"  Dominant spend categories: {top2[0]} and {top2[1]} "
            f"(means: {mean_profile.loc[cluster_id, top2[0]]:.0f} and "
            f"{mean_profile.loc[cluster_id, top2[1]]:.0f}).\n"
            f"  Channel composition: {majority_channel_pct:.1f}% {majority_channel}.\n"
            f"  Mean total annual spend: {total_spend[cluster_id]:,.0f} "
            f"({'highest' if spend_level == 'higher-total-spend' else 'lower'} of the two clusters).\n"
        )
        interpretations[cluster_id] = interpretation
        print(interpretation)

    print("[NOTE] Names above were generated mechanically from each cluster's "
          "actual top-2 spend categories and majority channel — not chosen "
          "before seeing the data, per the plan's requirement.")

    return {"cluster_names": cluster_names, "interpretations": interpretations}

## SECTION 3: Business Implications (Step 25)

In [ ]:
# SECTION 3: Business Implications (Step 25)
# ===========================================================================
def derive_business_implications(stats: dict, naming: dict) -> str:
    """State business implications tied explicitly to the actual named
    clusters and their spend/channel profiles, not generic boilerplate."""
    print("\n" + "=" * 70)
    print("SECTION 3: BUSINESS IMPLICATIONS")
    print("=" * 70)

    names = naming["cluster_names"]
    mean_profile = stats["mean_profile"]
    high_cluster = stats["total_spend"].idxmax()
    low_cluster = stats["total_spend"].idxmin()

    implications = (
        f"1. TARGETED PROMOTIONS:\n"
        f"   - Cluster {high_cluster} (\"{names[high_cluster]}\") has its highest mean spend in "
        f"{mean_profile.loc[high_cluster].idxmax()}. This supports a testable hypothesis that bundled offers "
        "involving this category and other high-spend categories may be relevant.\n"
        f"   - Cluster {low_cluster} (\"{names[low_cluster]}\") is comparatively Fresh/Frozen-heavy. "
        "A testable hypothesis is that category-specific offers may be more relevant than broad dry-goods promotions.\n\n"
        "2. INVENTORY & DELIVERY SERVICE PLANNING:\n"
        f"   - Cluster {low_cluster} is predominantly Horeca and Fresh/Frozen-heavy. The profile supports testing "
        "whether perishable-focused service options or different delivery frequencies improve operational fit. "
        "The dataset does not contain delivery-frequency data, so this is a hypothesis rather than a finding.\n"
        f"   - Cluster {high_cluster} is Retail-dominant and Grocery/Detergents-heavy. This supports testing whether "
        "larger-batch replenishment or category bundles are operationally effective.\n\n"
        "3. DIFFERENTIATED COMMERCIAL STRATEGIES:\n"
        f"   - Cluster {high_cluster} has {stats['total_spend'][high_cluster]:,.0f} mean annual spend versus "
        f"{stats['total_spend'][low_cluster]:,.0f} for Cluster {low_cluster}. The higher-spend segment could be "
        "evaluated for volume-based pricing or loyalty programs, while the lower-spend segment could be tested "
        "with category-specific offers. These are hypotheses requiring business validation.\n\n"
        "[CAVEAT] The dataset supports segmentation hypotheses, not causal claims about promotion response, "
        "delivery frequency, pricing sensitivity, or future sales. Operational decisions should be validated with "
        "order-level and experimental data."
    )
    print(implications)
    return implications

## SECTION 4: Limitations (Step 26)

In [ ]:
# SECTION 4: Limitations (Step 26)
# ===========================================================================
def document_limitations() -> str:
    """State the analysis's limitations plainly, including ones specific to
    this dataset/pipeline (not generic ML boilerplate)."""
    print("\n" + "=" * 70)
    print("SECTION 4: LIMITATIONS")
    print("=" * 70)

    limitations = (
        "1. SMALL DATASET: only 440 customer records. Cluster boundaries and "
        "profile statistics (means, ANOVA F-values) are estimated from a "
        "relatively small sample and may not generalize to a larger customer "
        "base without re-validation.\n\n"
        "2. SINGLE TIME-SNAPSHOT DATA: spend values are annual totals with no "
        "seasonality or time dimension. A customer's segment could shift "
        "across a year (e.g. a Horeca account's Fresh spend may spike "
        "seasonally) — this analysis cannot detect or account for that.\n\n"
        "3. NO DEMOGRAPHIC/FIRMOGRAPHIC DATA: beyond Channel and Region, no "
        "other customer attributes (business size, years as customer, order "
        "frequency, etc.) are available, limiting how deeply the 'why' behind "
        "each cluster's spend pattern can be explained.\n\n"
        "4. K-MEANS' SPHERICAL-CLUSTER ASSUMPTION: K-Means is most naturally suited to compact, roughly spherical "
        "clusters under Euclidean distance. The observed structure may not perfectly meet this assumption. The "
        "dendrogram (Phase 5) and residual skew after log-transform "
        "(Phase 4) suggest the true spend distribution may not perfectly fit "
        "this assumption, so some customers near the cluster boundary "
        "(visible in Phase 8's PCA scatter) may be ambiguously assigned.\n\n"
        "5. LOG-TRANSFORM INTERPRETABILITY: the log1p transform (Phase 4) "
        "was necessary to stabilize variance for clustering, but it changes "
        "the geometry of distances — a given gap in log-space does not "
        "correspond to a constant dollar gap in raw spend. Business "
        "stakeholders reading 'distance between clusters' should refer to "
        "the raw mean/median tables (Phase 7), not transformed-space "
        "distances, for dollar-denominated conclusions.\n\n"
        "6. TWO-CLUSTER SOLUTION IS COARSE: K=2 is the selected stable solution, but its low silhouette relative to an ideal separation "
        "and strong Channel alignment mean it may capture a broad business split rather than multiple fine-grained segments. "
        "This means the segmentation is necessarily coarse — it captures the "
        "dominant Horeca-vs-Retail-like spend split but may be masking finer "
        "sub-segments that this dataset's size/structure cannot reliably "
        "resolve with K-Means."
    )
    print(limitations)
    return limitations

## SECTION 5: Conclusion & Recommended Next Steps (Step 27)

In [ ]:
# SECTION 5: Conclusion & Recommended Next Steps (Step 27)
# ===========================================================================
def write_conclusion(stats: dict, naming: dict) -> str:
    """Summarize the key segments found and recommend concrete next steps,
    tied to the limitations above."""
    print("\n" + "=" * 70)
    print("SECTION 5: CONCLUSION & RECOMMENDED NEXT STEPS")
    print("=" * 70)

    names = naming["cluster_names"]
    primary_stability = stats.get("stability_ari", {}).get(2, None) if isinstance(stats, dict) else None
    conclusion = (
        "This analysis segmented 440 wholesale customers into 2 stable K-Means clusters based on annual spend "
        "across 6 product categories. The selected K=2 solution had the highest silhouette score in the tested "
        "range and remained robust under the project's seed-variation and bootstrap stability checks.\n\n"
        + "\n".join(f"  - Cluster {cid}: \"{name}\"" for cid, name in names.items())
        + "\n\nThe clusters align strongly with the pre-existing Channel variable even though Channel was excluded from "
        "clustering. This is best treated as post-hoc contextual evidence of business relevance, not external validation. "
        "The solution should therefore be interpreted as a broad spend segmentation that substantially separates "
        "Retail-dominant and Horeca-dominant purchasing patterns.\n\n"
        "RECOMMENDED NEXT STEPS:\n"
        "  1. Re-run the pipeline on more recent and larger customer data, ideally with monthly or transaction-level "
        "history, to test whether the segmentation persists over time.\n"
        "  2. Validate the business hypotheses using order frequency, delivery, promotion-response, margin, and customer-tenure data.\n"
        "  3. Investigate finer segmentation with additional algorithms only after validating whether K>2 solutions remain stable.\n"
        "  4. Treat the current segments as decision-support hypotheses rather than fixed customer categories."
    )
    print(conclusion)
    return conclusion

## SECTION 6: Assemble Report Section (Markdown Output)

In [ ]:
# SECTION 6: Assemble Report Section (Markdown Output)
# ===========================================================================
def assemble_report_section(naming: dict, implications: str, limitations: str, conclusion: str) -> None:
    """Write the interpretation, implications, limitations, and conclusion
    into a single markdown section for the final report assembly (Phase 10)."""
    print("\n" + "=" * 70)
    print("SECTION 6: ASSEMBLE REPORT SECTION (MARKDOWN)")
    print("=" * 70)

    interpretations_text = "\n\n".join(naming["interpretations"].values())

    content = f"""# 9. Interpretation, Business Implications & Limitations

## 9.1 Cluster Interpretation

{interpretations_text}

## 9.2 Business Implications

{implications}

## 9.3 Limitations

{limitations}

## 9.4 Conclusion & Recommended Next Steps

{conclusion}
"""

    out_path = "section9_interpretation.md"
    with open(out_path, "w") as f:
        f.write(content)
    print(f"[OK] Saved {out_path} ({len(content)} characters)")

## MAIN — run Phase 9 end to end

In [ ]:
# MAIN — run Phase 9 end to end
# ===========================================================================
if __name__ == "__main__":
    stats = load_and_recompute_stats()
    naming = name_and_interpret_clusters(stats)
    implications = derive_business_implications(stats, naming)
    limitations = document_limitations()
    conclusion = write_conclusion(stats, naming)
    assemble_report_section(naming, implications, limitations, conclusion)

    print("\n" + "=" * 70)
    print("PHASE 9 COMPLETE")
    print("=" * 70)
    print(f"[OK] Clusters named/interpreted from actual data: {naming['cluster_names']}")
    print("[OK] Business implications derived and tied to specific cluster findings.")
    print("[OK] Limitations documented (dataset size, time-snapshot, K-Means geometry, log-transform interpretability, K=2 coarseness).")
    print("[OK] Conclusion and next steps written.")
    print("[OK] Report section saved to section9_interpretation.md.")
    print("[OK] Ready for Phase 10 (Final Report Assembly).")

### Phase 9 checkpoint

Review the outputs and figures generated by this phase before moving to the next phase.